In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from collections import defaultdict
from tqdm import tqdm

# Global configuration for matplotlib plots
SMALL_SIZE = 12
MEDIUM_SIZE = 16
BIGGER_SIZE = 20

plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=BIGGER_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

# Fix random generator for reproducibility
rng = np.random.default_rng(0)

# Load datasets
airfield_stats_test = pd.read_csv("airfield_statistics_test.csv")
airfield_stats_train = pd.read_csv("airfield_statistics_train.csv")

X_train_origin = airfield_stats_train.iloc[:, :6].values
y_train_origin = airfield_stats_train.iloc[:, 6].values

X_train = X_train_origin.copy()
y_train = y_train_origin.copy()

X_test_origin = airfield_stats_test.iloc[:, :6].values
y_test_origin = airfield_stats_test.iloc[:, 6].values

X_test = X_test_origin.copy()
y_test = y_test_origin.copy()

cat_columns_dict = {i: False for i in range(X_train.shape[1])}
feature_names = airfield_stats_train.columns[:6].tolist()


# Dense (fully-connected) layer operation.
def dense(X, W, b):
    # X: (K, h_in), W: (h_out, h_in), b: (h_out,)
    a = np.vstack([W @ x + b for x in X])
    return a


# ReLU activation function.
def relu_activation(a):
    h = np.maximum(a, 0.0)
    return h


# Gradient of ReLU activation.
def grad_relu_activation(a):
    grad = np.zeros_like(a)
    grad[a > 0] = 1
    return grad


# Lookup table for activation functions.
activation_table = {"relu": relu_activation, "identity": lambda x: x}

# Lookup table for gradients of activation functions.
grad_activation_table = {
    "relu": grad_relu_activation,
    "identity": lambda x: np.ones_like(x),
}


# Revised MLP class.
class MLP:
    def __init__(self, seed=42):
        self.layers = []  # each layer is a dict with keys: W, b, activation, index.
        self.rng = np.random.default_rng(seed)

    def n_parameters(self):
        return sum(l["W"].size + l["b"].size for l in self.layers)

    def n_layers(self):
        return len(self.layers)

    def add_layer(self, in_dim, out_dim, activation="identity"):
        self.layers.append(
            {
                "index": len(self.layers),
                "W": self.rng.normal(size=(out_dim, in_dim))
                * np.sqrt(2.0 / (in_dim + out_dim)),
                "b": np.zeros(out_dim),
                "activation": activation,
            }
        )

    def predict(self, X):
        if X.ndim == 1:
            X = X.reshape(1, -1)
        h = X
        forward_pass = []
        for i, layer in enumerate(self.layers):
            a = dense(h, layer["W"], layer["b"])
            h = activation_table[layer["activation"]](a)
            forward_pass.append({"index": i, "a": a, "h": h})
        y_hat = h
        return y_hat, forward_pass


# Mean Squared Error loss.
def mse_loss(y_true, y_pred):
    assert (
        y_true.size == y_pred.size
    ), "Ground-truth and predictions have different dimensions."
    y_true = y_true.reshape(y_pred.shape)
    loss = np.mean((y_true - y_pred) ** 2, keepdims=True)
    return loss


# Compound loss function.
def compound_loss(T, T_pred, l=0):
    T = T.reshape(T_pred.shape)
    T_min, T_max = T[:, 0], T[:, 1]
    T_min_pred, T_max_pred = T_pred[:, 0], T_pred[:, 1]

    MSE_min = mse_loss(T_min, T_min_pred)
    MSE_max = mse_loss(T_max, T_max_pred)

    key = T_min_pred > T_max_pred  # boolean array; no list wrapping.
    return MSE_min + MSE_max + l * np.mean(key)


# Gradient of compound loss.
def compound_grad(T, T_pred, l=0):
    assert T.shape == T_pred.shape, "Error: Shapes of T and T_pred must match!"
    T_min, T_max = T[:, 0], T[:, 1]
    T_min_pred, T_max_pred = T_pred[:, 0], T_pred[:, 1]

    grad_min = (2.0 * (T_min - T_min_pred) / len(T_min)).reshape(-1, 1)
    grad_max = (2.0 * (T_max - T_max_pred) / len(T_min)).reshape(-1, 1)

    key = (T_min_pred > T_max_pred).astype(float).reshape(-1, 1)
    grad_min += (l * key) / len(T_min)
    grad_max -= (l * key) / len(T_min)

    return np.hstack((grad_min, grad_max))


# Test compound_grad to check shapes.
batch_size = 5
T = np.array([[1.0, 3.0], [2.0, 5.0], [0.5, 2.5], [4.0, 6.0], [3.0, 7.0]])
T_pred = np.array([[1.2, 2.8], [1.8, 5.2], [0.4, 2.4], [4.1, 5.9], [2.9, 6.8]])
l_penalty = 0.1
gradients = compound_grad(T, T_pred, l_penalty)
print("T:")
print(T)
print("\nT_pred:")
print(T_pred)
print("\nComputed Gradients:")
print(gradients)
expected_shape = (batch_size, 2)
print("\nExpected Shape:", expected_shape)
print("Actual Shape:  ", gradients.shape)
assert (
    gradients.shape == expected_shape
), f"Shape mismatch! Expected {expected_shape}, but got {gradients.shape}."
print("\n✅ Test Passed: Gradients shape is correct!")


# Backpropagation function.
def backpropagate(layers, forward_pass, delta_output):
    gradients = []
    delta = delta_output
    assert len(layers) == len(
        forward_pass
    ), "Mismatch between number of layers and forward computations."

    # Iterate backward over layers.
    for layer, forward_comp in reversed(list(zip(layers, forward_pass))):
        h = forward_comp["h"]
        gradients.append({"W": (delta.T @ h) / h.shape[0], "b": delta.mean(axis=0)})
        grad_activation_f = grad_activation_table[layer["activation"]]
        grad_activation = grad_activation_f(forward_comp["a"])
        # Correct multiplication order:
        delta = np.stack(
            [
                layer["W"].T @ (np.diag(gi) @ di)
                for gi, di in zip(grad_activation, delta)
            ]
        )
    gradients = list(reversed(gradients))
    return gradients


# Single SGD update step.
def sgd_step(X, y, mlp, lam=0, learning_rate=1e-3):
    y_hat, forward_pass = mlp.predict(X)
    delta_output = compound_grad(y, y_hat, lam)
    gradients = backpropagate(mlp.layers, forward_pass, delta_output)

    updated_layers = []
    for layer, grad in zip(mlp.layers, gradients):
        W = layer["W"] - learning_rate * grad["W"]
        b = layer["b"] - learning_rate * grad["b"]
        updated_layers.append(
            {"W": W, "b": b, "activation": layer["activation"], "index": layer["index"]}
        )
    return updated_layers


# R^2 score for regression.
def r2_score(y, y_hat):
    y = y.reshape(y_hat.shape)
    y_bar = y.mean()
    ss_tot = ((y - y_bar) ** 2).sum()
    ss_res = ((y - y_hat) ** 2).sum()
    return 1 - (ss_res / ss_tot)


# Stochastic Gradient Descent training.
def sgd(
    X_train,
    y_train,
    X_test,
    y_test,
    mlp,
    lam=0,
    learning_rate=1e-3,
    n_epochs=10,
    minibatchsize=1,
    seed=42,
):
    rng = np.random.default_rng(seed)
    n_iterations = int(len(y_train) / minibatchsize)
    losses_train = []
    losses_test = []
    epochs_bar = tqdm(range(n_epochs), desc="Epochs")

    for epoch in epochs_bar:
        p = rng.permutation(len(y_train))
        X_train_shuffled = X_train[p]
        y_train_shuffled = y_train[p]

        for j in range(n_iterations):
            X_batch = X_train_shuffled[j * minibatchsize : (j + 1) * minibatchsize]
            y_batch = y_train_shuffled[j * minibatchsize : (j + 1) * minibatchsize]
            updated_layers = sgd_step(X_batch, y_batch, mlp, lam, learning_rate)
            mlp.layers = updated_layers

        y_hat_train, _ = mlp.predict(X_train)
        loss_train = compound_loss(y_train, y_hat_train, l=lam).squeeze()
        losses_train.append(loss_train)
        y_hat_test, _ = mlp.predict(X_test)
        loss_test = compound_loss(y_test, y_hat_test, l=lam).squeeze()
        losses_test.append(loss_test)
        epochs_bar.set_description(
            f"train_loss: {loss_train:.2f}, test_loss: {loss_test:.2f}, "
            f"train_R^2: {r2_score(y_train, y_hat_train):.2f}, "
            f"test_R^2: {r2_score(y_test, y_hat_test):.2f}"
        )
    return mlp, losses_train, losses_test


# Prepare target data.
runway_train = np.array(airfield_stats_train.iloc[:, 7:9].values)
runway_test = np.array(airfield_stats_test.iloc[:, 7:9].values)

# Build the MLP architecture.
mlp = MLP(seed=2)
# First layer: transform input (6-dim) to 20-dim, using identity activation.
mlp.add_layer(X_train.shape[1], 20, activation="identity")
# Hidden layer.
mlp.add_layer(20, 20, activation="relu")
# Output layer: 20-dim to 2-dim with identity activation (for regression).
mlp.add_layer(20, 2, activation="identity")

print("Number of layers:", mlp.n_layers())
print("Number of trainable parameters:", mlp.n_parameters())

# Train the MLP using SGD.
mlp, losses_train, losses_test = sgd(
    X_train,
    runway_train,
    X_test,
    runway_test,
    mlp,
    lam=1,
    learning_rate=5e-5,
    n_epochs=200,
    minibatchsize=20,
)

# Plot training progress.
fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(np.arange(1, 201), losses_train, label="Train")
ax.plot(np.arange(1, 201), losses_test, label="Test")
ax.set(title="Losses vs Epochs", xlabel="Epoch", ylabel="Compound Loss")
ax.legend()
plt.show()

T:
[[1.  3. ]
 [2.  5. ]
 [0.5 2.5]
 [4.  6. ]
 [3.  7. ]]

T_pred:
[[1.2 2.8]
 [1.8 5.2]
 [0.4 2.4]
 [4.1 5.9]
 [2.9 6.8]]

Computed Gradients:
[[-0.08  0.08]
 [ 0.08 -0.08]
 [ 0.04  0.04]
 [-0.04  0.04]
 [ 0.04  0.08]]

Expected Shape: (5, 2)
Actual Shape:   (5, 2)

✅ Test Passed: Gradients shape is correct!
Number of layers: 3
Number of trainable parameters: 602


Epochs:   0%|          | 0/200 [00:00<?, ?it/s]


ValueError: operands could not be broadcast together with shapes (20,6) (20,20) 